# 06 - Hydration Signal Model (EfficientNet-B0 + Handcrafted Features)

Train a Gabor + LBP hydration scorer for the **hydration** skin signal.

## Features
- Gabor filter bank (4 orientations x 3 frequencies x 2 stats = 24 features)
- LBP (Local Binary Pattern) histogram (18 bins)
- Specular highlight distribution (2 features: ratio + spread)
- Total: 44-dim handcrafted feature vector

## Architecture
- Backbone: EfficientNet-B0 (pretrained ImageNet) -> 1280-dim CNN features
- Handcrafted branch: FC(44 -> 64)
- Fusion: concat(1280 + 64) -> FC(1344 -> 256 -> 1)
- Regression head for hydration score [0-100]

## Training
- MSE loss, AdamW lr=1e-4, CosineAnnealing, 30 epochs, batch=32

## Output
- Dual-input ONNX model (image + handcrafted_features) for backend inference

In [ ]:
# Install dependencies (Colab)
!pip install -q torch torchvision timm albumentations onnx onnxruntime opencv-python scikit-image

In [ ]:
# Mount Google Drive for data access and saving results
import os

IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/cornell-hackathon/ml'
    DATA_BASE = os.path.join(DRIVE_BASE, 'data', 'hydration')
    SAVE_DIR = os.path.join(DRIVE_BASE, 'models', 'hydration')
else:
    # Local / VM paths
    DATA_BASE = '/root/cornell-hackathon/ml/data/hydration'
    SAVE_DIR = '/root/cornell-hackathon/ml/models/hydration'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Data directory: {DATA_BASE}')
print(f'Save directory: {SAVE_DIR}')

In [ ]:
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.feature import local_binary_pattern
from tqdm import tqdm
import matplotlib.pyplot as plt
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Feature Extraction

Gabor filters capture texture at multiple orientations and frequencies. LBP captures
micro-texture patterns. Specular highlights indicate skin moisture level.

**Note:** The prepared dataset already includes pre-computed handcrafted features (44-dim) in
each annotation, so runtime feature extraction is only needed for inference on new images.

In [ ]:
def build_gabor_bank(orientations: int = 4, frequencies: list = [0.1, 0.25, 0.4]) -> list:
    """Build Gabor filter bank: 4 orientations x 3 frequencies = 12 filters."""
    kernels = []
    for theta_idx in range(orientations):
        theta = theta_idx * np.pi / orientations
        for freq in frequencies:
            kernel = cv2.getGaborKernel(
                ksize=(31, 31), sigma=4.0, theta=theta,
                lambd=1.0 / freq, gamma=0.5, psi=0
            )
            kernels.append(kernel)
    return kernels


def extract_gabor_features(gray: np.ndarray, kernels: list) -> np.ndarray:
    """Apply Gabor bank and compute mean + std energy per filter."""
    features = []
    for kernel in kernels:
        response = cv2.filter2D(gray, cv2.CV_64F, kernel)
        features.extend([response.mean(), response.std()])
    return np.array(features, dtype=np.float32)


def extract_lbp_histogram(gray: np.ndarray, radius: int = 2, n_points: int = 16) -> np.ndarray:
    """Compute uniform LBP histogram."""
    lbp = local_binary_pattern(gray, n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=n_points + 2, range=(0, n_points + 2), density=True)
    return hist.astype(np.float32)


def extract_specular_features(image: np.ndarray, threshold: int = 220) -> np.ndarray:
    """Analyze specular highlight distribution as moisture indicator."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    highlights = (gray > threshold).astype(np.float32)
    highlight_ratio = highlights.mean()
    # Spatial spread of highlights
    if highlights.sum() > 0:
        coords = np.argwhere(highlights > 0)
        spread = coords.std(axis=0).mean() / max(gray.shape)
    else:
        spread = 0.0
    return np.array([highlight_ratio, spread], dtype=np.float32)


GABOR_BANK = build_gabor_bank()
HANDCRAFTED_DIM = len(GABOR_BANK) * 2 + 18 + 2  # gabor(24) + lbp(18) + specular(2) = 44
print(f'Handcrafted feature dimension: {HANDCRAFTED_DIM}')

## Dataset

Loads from prepared annotation JSONs (list format) with train/val/test splits.
Each annotation: `{image, hydration_score, handcrafted_features: [44 floats]}`

Uses pre-computed handcrafted features from annotations for training speed.

In [ ]:
class HydrationDataset(Dataset):
    """Dataset for hydration signal training with pre-computed handcrafted features."""

    def __init__(self, image_dir: str, annotations_path: str, transform=None):
        self.image_dir = image_dir
        with open(annotations_path) as f:
            self.annotations = json.load(f)
        self.transform = transform
        # Filter out entries where image doesn't exist
        valid = []
        for ann in self.annotations:
            img_path = os.path.join(self.image_dir, ann['image'])
            if os.path.exists(img_path):
                valid.append(ann)
        print(f'Loaded {len(valid)}/{len(self.annotations)} valid samples from {annotations_path}')
        self.annotations = valid

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]

        image = cv2.imread(os.path.join(self.image_dir, ann['image']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image, (224, 224))

        # Use pre-computed handcrafted features from annotations
        handcrafted = np.array(ann['handcrafted_features'], dtype=np.float32)

        if self.transform:
            augmented = self.transform(image=image_resized)
            image_tensor = augmented['image']
        else:
            image_tensor = torch.from_numpy(image_resized.transpose(2, 0, 1)).float() / 255.0

        label = torch.tensor([ann['hydration_score']], dtype=torch.float32)
        return image_tensor, torch.from_numpy(handcrafted), label


train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [ ]:
# Load datasets
IMAGE_DIR = os.path.join(DATA_BASE, 'images')
TRAIN_ANN = os.path.join(DATA_BASE, 'annotations', 'train.json')
VAL_ANN = os.path.join(DATA_BASE, 'annotations', 'val.json')
TEST_ANN = os.path.join(DATA_BASE, 'annotations', 'test.json')

print(f'Image dir: {IMAGE_DIR}')
print(f'Train annotations: {TRAIN_ANN}')

train_ds = HydrationDataset(IMAGE_DIR, TRAIN_ANN, transform=train_transform)
val_ds = HydrationDataset(IMAGE_DIR, VAL_ANN, transform=val_transform)
test_ds = HydrationDataset(IMAGE_DIR, TEST_ANN, transform=val_transform)

print(f'\nDataset sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

# Quick sanity check
img, hc, lbl = train_ds[0]
print(f'Image shape: {img.shape}, Handcrafted shape: {hc.shape}, Label: {lbl.item():.1f}')

## Model Definition

EfficientNet-B0 backbone with feature fusion. CNN features (1280-dim) are concatenated with
handcrafted Gabor/LBP/specular features (44->64-dim) before the regression head.

In [ ]:
class HydrationModel(nn.Module):
    """Hydration signal model with EfficientNet-B0 + handcrafted feature fusion."""

    def __init__(self, handcrafted_dim: int = 44, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        cnn_dim = self.backbone.num_features  # 1280

        self.handcrafted_fc = nn.Sequential(
            nn.Linear(handcrafted_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.head = nn.Sequential(
            nn.Linear(cnn_dim + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, image, handcrafted):
        cnn_feat = self.backbone(image)
        hc_feat = self.handcrafted_fc(handcrafted)
        fused = torch.cat([cnn_feat, hc_feat], dim=-1)
        return self.head(fused)


model = HydrationModel(handcrafted_dim=HANDCRAFTED_DIM, pretrained=True).to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## Training Loop

In [ ]:
# Hyperparameters
NUM_EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-4


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for images, handcrafted, labels in tqdm(loader, desc='Training'):
        images = images.to(DEVICE)
        handcrafted = handcrafted.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        preds = model(images, handcrafted)
        loss = nn.functional.mse_loss(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, handcrafted, labels in loader:
        images, handcrafted = images.to(DEVICE), handcrafted.to(DEVICE)
        preds = model(images, handcrafted)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
    preds = torch.cat(all_preds).squeeze()
    labels = torch.cat(all_labels).squeeze()
    mae = torch.abs(preds - labels).mean().item()
    # Pearson correlation
    if preds.numel() > 1:
        pearson = torch.corrcoef(torch.stack([preds, labels]))[0, 1].item()
    else:
        pearson = 0.0
    return {'mae': mae, 'pearson_r': pearson}

In [ ]:
# Create data loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Optimizer and scheduler
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f'Training for {NUM_EPOCHS} epochs with batch_size={BATCH_SIZE}, lr={LR}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Training loop
best_mae = float('inf')
history = {'train_loss': [], 'val_mae': [], 'val_pearson': []}

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer)
    metrics = evaluate(model, val_loader)
    scheduler.step()

    # Track history
    history['train_loss'].append(train_loss)
    history['val_mae'].append(metrics['mae'])
    history['val_pearson'].append(metrics['pearson_r'])

    elapsed = time.time() - t0

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | Loss: {train_loss:.4f} | "
          f"MAE: {metrics['mae']:.2f} | Pearson r: {metrics['pearson_r']:.4f} | "
          f"LR: {scheduler.get_last_lr()[0]:.6f}")

    if metrics['mae'] < best_mae:
        best_mae = metrics['mae']
        best_path = os.path.join(SAVE_DIR, 'best_hydration_model.pt')
        torch.save(model.state_dict(), best_path)
        print(f'  -> Saved best model (MAE: {best_mae:.2f}) to {best_path}')

print(f'\nTraining complete. Best MAE: {best_mae:.2f}')

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', color='blue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curve
axes[1].plot(history['val_mae'], label='Val MAE', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Validation MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Pearson R curve
axes[2].plot(history['val_pearson'], label='Val Pearson r', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Pearson r')
axes[2].set_title('Validation Correlation')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(fig_path, dpi=150)
plt.show()
print(f'Saved training curves to {fig_path}')

## Test Set Evaluation

In [ ]:
# Load best model and evaluate on test set
best_path = os.path.join(SAVE_DIR, 'best_hydration_model.pt')
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
print(f'Loaded best model from {best_path}')

test_metrics = evaluate(model, test_loader)
print(f"\nTest Results:")
print(f"  Hydration MAE: {test_metrics['mae']:.2f}")
print(f"  Pearson r:     {test_metrics['pearson_r']:.4f}")

# Save test metrics
metrics_path = os.path.join(SAVE_DIR, 'test_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f'\nSaved test metrics to {metrics_path}')

## ONNX Export

Export the trained model to ONNX format with dual inputs (image + handcrafted features)
for backend inference.

In [ ]:
def export_to_onnx(model, output_path):
    model.eval()
    dummy_image = torch.randn(1, 3, 224, 224).to(DEVICE)
    dummy_hc = torch.randn(1, HANDCRAFTED_DIM).to(DEVICE)

    torch.onnx.export(
        model,
        (dummy_image, dummy_hc),
        output_path,
        input_names=['image', 'handcrafted_features'],
        output_names=['hydration_score'],
        dynamic_axes={
            'image': {0: 'batch'},
            'handcrafted_features': {0: 'batch'},
            'hydration_score': {0: 'batch'},
        },
        opset_version=17,
    )
    print(f'Exported ONNX model to {output_path}')

    # Verify
    import onnxruntime as ort
    session = ort.InferenceSession(output_path)
    result = session.run(None, {
        'image': dummy_image.cpu().numpy(),
        'handcrafted_features': dummy_hc.cpu().numpy(),
    })
    print(f'ONNX output shape: {result[0].shape}, sample: {result[0][0][0]:.2f}')
    print(f'Input names: {[inp.name for inp in session.get_inputs()]}')
    print(f'Output names: {[out.name for out in session.get_outputs()]}')
    return session


onnx_path = os.path.join(SAVE_DIR, 'hydration_model.onnx')
ort_session = export_to_onnx(model, onnx_path)
print(f'\nONNX model size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB')

## Inference Example

In [ ]:
# Quick inference test with a sample image
import onnxruntime as ort

sample_img, sample_hc, sample_label = test_ds[0]
sample_img_np = sample_img.unsqueeze(0).numpy()
sample_hc_np = sample_hc.unsqueeze(0).numpy()

ort_session = ort.InferenceSession(onnx_path)
ort_result = ort_session.run(None, {
    'image': sample_img_np,
    'handcrafted_features': sample_hc_np,
})

print('ONNX Inference on sample image:')
print(f'  Predicted hydration score: {ort_result[0][0][0]:.1f}')
print(f'  Ground truth:              {sample_label.item():.1f}')

## Inference Helper: Full Pipeline for New Images

For inference on new images where handcrafted features aren't pre-computed.

In [ ]:
def predict_hydration(image_path: str, ort_session) -> float:
    """Full inference pipeline: image -> handcrafted features + CNN -> hydration score."""
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image, (224, 224))

    # Extract handcrafted features
    gray = cv2.cvtColor(image_resized, cv2.COLOR_RGB2GRAY)
    gabor_feat = extract_gabor_features(gray, GABOR_BANK)
    lbp_feat = extract_lbp_histogram(gray)
    specular_feat = extract_specular_features(image_resized)
    handcrafted = np.concatenate([gabor_feat, lbp_feat, specular_feat])

    # Normalize image
    img_normalized = image_resized.astype(np.float32) / 255.0
    img_normalized = (img_normalized - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    img_tensor = img_normalized.transpose(2, 0, 1)[np.newaxis]  # (1, 3, 224, 224)

    result = ort_session.run(None, {
        'image': img_tensor.astype(np.float32),
        'handcrafted_features': handcrafted[np.newaxis],
    })
    return result[0][0][0]


# Test with a sample image from dataset
sample_image_path = os.path.join(IMAGE_DIR, test_ds.annotations[0]['image'])
score = predict_hydration(sample_image_path, ort_session)
print(f'Full pipeline prediction: {score:.1f}')
print(f'Ground truth: {test_ds.annotations[0]["hydration_score"]:.1f}')

In [ ]:
# Summary of saved artifacts
print('=== Saved Artifacts ===')
for fname in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f'  {fname}: {size_mb:.1f} MB')
print(f'\nAll artifacts saved to: {SAVE_DIR}')